# Example 5 - Pine Island Glacier: Inversion sensitivity analysis

In [Example 2](ex2_PineIslandGlacier.ipynb) we inferred the basal friction coefficient beneath Pine Island Glacier (PIG) using a single, hand-picked set of cost function coefficients. In practice, the inverted field depends strongly on how the cost functions are weighted, and choosing these weights is one of the most important (and most subjective) steps in setting up an inversion.

This notebook demonstrates the `pyissm.inversion` sub-package, which provides tools to make that choice systematically. We will:

1. Assign cost function coefficients, including spatial masks, using `pyissm.inversion.sensitivity.assign_cost_functions()`
2. Run a single inversion and inspect it with the diagnostics in `pyissm.inversion.metrics`
3. Build and run a sweep of regularisation coefficients and use an **L-curve** to choose a value
4. Build and run a two-dimensional sweep of the misfit coefficients and compare runs using **sensitivity heatmaps**
5. Rank runs with normalised diagnostics and load the selected run for further use

**NOTE: This notebook continues from Example 2 and requires the parameterised model `ex2_PIG_param.nc` saved in Section 3 of that notebook. Please complete Example 2 (at least up to and including Section 3) before running this notebook.**

**NOTE: Sections 3 and 4 run 15 inversions in total. Each inversion takes a few minutes on 2 CPUs at the Example 2 mesh resolution. You can reduce the number of coefficient values tested to shorten the run time.**

In [ ]:
import os
import pyissm
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt

---
## Setup your modelling environment

As in Example 2, this notebook assumes that pyISSM (and therefore, this notebook) is installed in `~/pyISSM/` (locally) or `/home/<group>/<user>/pyISSM/` (on NCI *Gadi*). We refer to this path as `pyissm_home`. Please update the following paths if required:

- `tutorial_dir = <PATH_TO_NOTEBOOK>` where this notebook (and the Example 2 model files) are located. By default, this is assumed to be `pyissm_home/tutorials`
- `execution_dir = <PATH_TO_DIRECTORY>` where ISSM stages files for model execution. You must have `rwx` permissions for this directory.
- `sensitivity_dir = <PATH_TO_DIRECTORY>` where pyISSM stores organised sensitivity experiment outputs (saved models, manifests, diagnostics and summary reports). You must have `rwx` permissions for this directory.

`execution_dir` and `sensitivity_dir` are intentionally kept separate: `md.cluster.executionpath` is where ISSM writes its runtime files, while `sensitivity_dir` is where pyISSM keeps a tidy record of each experiment.

In [ ]:
## Set required paths
tutorial_dir = str(Path.home() / 'pyISSM' / 'tutorials')
execution_dir = tutorial_dir + '/models'
sensitivity_dir = tutorial_dir + '/ex5_sensitivity'

# Check that the directories exist. If not, create them
for d in [execution_dir, sensitivity_dir]:
    os.makedirs(d, exist_ok = True)

# Print the paths for visibility
print(f"The following `tutorial_dir` is set: {tutorial_dir}")
print(f"The following `execution_dir` is set: {execution_dir}")
print(f"The following `sensitivity_dir` is set: {sensitivity_dir}")

---
## 1. Configure the inversion

We start from the parameterised (but not yet inverted) model from Example 2, and define the parts of the inversion that will stay fixed throughout this notebook: the control parameter (the Budd `FrictionCoefficient`), its bounds, and the number of optimisation steps.

We use the same three cost functions as Example 2:

| ID | Name | Role |
|----|------|------|
| 101 | `SurfaceAbsVelMisfit` | Misfit — dominated by fast-flowing ice |
| 103 | `SurfaceLogVelMisfit` | Misfit — gives more weight to slow-flowing ice |
| 501 | `DragCoefficientAbsGradient` | Regularisation (Tikhonov) — penalises rough friction fields |

The cost functions and their coefficients are what we will vary, so we do not set them here.

In [ ]:
# Load the parameterised model from Example 2
md = pyissm.model.io.load_model(tutorial_dir + '/ex2_PIG_param.nc')

# Fixed inversion settings
md.inversion.iscontrol = 1
md.inversion.control_parameters = ['FrictionCoefficient']
md.inversion.min_parameters = 1 * np.ones(md.mesh.numberofvertices, )
md.inversion.max_parameters = 200 * np.ones(md.mesh.numberofvertices, )

# Verbosity
md.verbose = md.verbose.deactivate_all()
md.verbose.control = 1

# Execution settings
md.cluster.np = 2
md.cluster.executionpath = execution_dir

---
## 2. A single inversion

### 2.1 Assigning cost functions with masks

In Example 2, we built `md.inversion.cost_functions_coefficients` by hand. `pyissm.inversion.sensitivity.assign_cost_functions()` does this from a dictionary that maps each cost function ID to a scalar coefficient, and sets `md.inversion.cost_functions` at the same time.

It also accepts boolean masks that switch coefficients off (set them to 0) in parts of the domain:

- `global_mask` applies to **all** cost functions. Where `True`, coefficients are applied; where `False`, they are set to 0.
- `coeff_masks` is a dictionary of masks that apply to **individual** cost functions, on top of `global_mask`.

Here, we only fit velocities where ice is present and an observation exists. We also show how to restrict an individual cost function: here, we apply the log-velocity misfit (103) only over grounded ice, so that the slow-flowing grounded regions it emphasises are not mixed with the ice shelf, where velocity depends mainly on ice rheology and buttressing rather than on basal friction.

In [ ]:
# Only fit where ice is present and velocity has been observed
global_mask = (md.mask.ice_levelset < 0) & np.isfinite(md.inversion.vel_obs) & (md.inversion.vel_obs > 0)

# Only apply the log-velocity misfit over grounded ice
coeff_masks = {103: md.mask.ocean_levelset > 0}

# Cost function coefficients (as used in Example 2)
coefficients = {101: 50,
                103: 1,
                501: 1e-19}

md = pyissm.inversion.sensitivity.assign_cost_functions(md,
                                                        coefficients,
                                                        global_mask = global_mask,
                                                        coeff_masks = coeff_masks)

print(md.inversion.cost_functions)
print(md.inversion.cost_functions_coefficients.shape)

In [ ]:
# Visualise the coefficient applied to each cost function
fig, axes = plt.subplots(1, 3, figsize = (15, 4.5), sharey = True, constrained_layout = True)
for i, (cf, ax) in enumerate(zip(md.inversion.cost_functions, axes)):
    pyissm.plot.plot_model_field(md,
                                 md.inversion.cost_functions_coefficients[:, i],
                                 ax = ax,
                                 show_cbar = True)
    ax.set_title(f'Cost function {cf} coefficient')
for ax in axes[1:]:
    ax.set_ylabel('')

### 2.2 Run the inversion

In [ ]:
md = pyissm.model.execute.solve(md, 'Stressbalance')

# Save the model
pyissm.model.io.save_model(md, tutorial_dir + '/ex5_PIG_single_inversion.nc')

### 2.3 Inversion diagnostics

`pyissm.inversion.metrics` provides functions to summarise the result of an inversion:

- `cost_function_values()` — the final value of each cost function and the total cost
- `cost_function_ratios()` — pairwise ratios between the final cost function values. These show which terms dominate the total cost.
- `velocity_residual_area_metrics()` — the area (and fraction of the domain) where the model is too fast (positive residual) or too slow (negative residual). A well-balanced inversion has similar positive and negative fractions.
- `field_smoothness_metrics()` — the area-weighted mean and RMSE of the gradient magnitude of the inverted field. Larger values indicate a rougher field.

We also use `pyissm.tools.diagnostics.velocity_rmse()` for the area-weighted velocity RMSE.

In [ ]:
diagnostics = {
    'vel_rmse': pyissm.tools.diagnostics.velocity_rmse(md),
    **pyissm.inversion.metrics.cost_function_values(md),
    **pyissm.inversion.metrics.cost_function_ratios(md),
    **pyissm.inversion.metrics.velocity_residual_area_metrics(md),
    **pyissm.inversion.metrics.field_smoothness_metrics(md),
}

pd.Series(diagnostics)

The convergence history records the value of each cost function at every optimisation step. Plotting it shows whether the inversion has converged (the total cost has flattened out) or would benefit from more steps (`md.inversion.nsteps`).

In [ ]:
convergence_history = pyissm.inversion.metrics.extract_convergence_history(md)

fig, ax = pyissm.inversion.plot.plot_convergence_history(convergence_history)
ax.set_title('Cost function convergence history')
ax.grid()

Finally, `pyissm.inversion.plot.plot_run_summary()` collects all of the above into a two-page PDF report:

- **Page 1:** observed velocity, modelled velocity, velocity residuals and the inverted field
- **Page 2:** convergence history, histograms of the velocity residuals and inverted-field gradient, and a text summary of the diagnostics

This is useful for keeping a record of each run, and is generated automatically for every run in a sensitivity study (Section 3).

In [ ]:
pyissm.inversion.plot.plot_run_summary(md, output_pdf = tutorial_dir + '/ex5_PIG_single_inversion_summary.pdf')

---
## 3. Choosing the regularisation coefficient: the L-curve

The regularisation coefficient (here, the coefficient for cost function 501) controls the trade-off between fitting the observations and keeping the friction field smooth:

- **Too small**: the friction field over-fits noise in the observations and becomes unrealistically rough.
- **Too large**: the friction field is very smooth, but the model fits the observations poorly.

A common way to choose this coefficient is the **L-curve**: run the inversion for a range of coefficients and plot the data misfit against the (unweighted) regularisation term on log-log axes. The curve typically has an "L" shape, and the coefficient at the corner gives a good balance between the two.

### 3.1 Build the parameter grid

`pyissm.inversion.sensitivity.build_parameter_grid()` takes a dictionary that maps each cost function ID to a **list** of coefficients and returns every combination as a table. Each row is given a `run_id` and a unique `run_name`. Here, we keep the misfit coefficients fixed and vary only the regularisation coefficient across six orders of magnitude.

In [ ]:
lcurve_coefficients = {101: [50],
                       103: [1],
                       501: [1e-22, 1e-21, 1e-20, 1e-19, 1e-18, 1e-17]}

lcurve_grid = pyissm.inversion.sensitivity.build_parameter_grid(lcurve_coefficients)
lcurve_grid

### 3.2 Run the sensitivity experiments

`pyissm.inversion.sensitivity.parameter_sensitivity()` loops over the parameter grid and, for each row, copies the model, assigns the cost functions (passing on any masks), solves the inversion, and saves the result to its own directory in `sensitivity_dir`. It returns a **manifest** describing every run (including its status), which is also saved to `sensitivity_dir/manifest.csv`.

If a run fails, it is marked as `"failed"` in the manifest and the remaining runs continue.

In [ ]:
lcurve_manifest = pyissm.inversion.sensitivity.parameter_sensitivity(md,
                                                                     lcurve_grid,
                                                                     output_dir = sensitivity_dir + '/lcurve',
                                                                     run = True,
                                                                     global_mask = global_mask,
                                                                     coeff_masks = coeff_masks)
lcurve_manifest

<div class="alert alert-info">

**Running on a cluster (e.g. NCI *Gadi*)**

On a cluster where jobs are submitted to a queue, you will usually want to submit all runs at once rather than wait for each one to finish. To do this, set `md.settings.waitonlock = 0` before calling `parameter_sensitivity()`. Each run is then submitted and marked as `"submitted"` in the manifest.

Once all jobs have finished, call `parameter_sensitivity()` again with the same arguments, but with `run = False` and `load_only = True`. This loads the results from the execution directory, saves each model to `sensitivity_dir`, and marks the runs as `"completed"`.

</div>

### 3.3 Compute diagnostics

`pyissm.inversion.sensitivity.compute_sensitivity_diagnostics()` loads each run in the manifest and computes the diagnostics from Section 2.3. It returns a table with one row per run, and also saves:

- `diagnostics.csv` to `output_dir` (updated after each run)
- each run's convergence history to `<run_dir>/<run_name>_convergence_history.csv`
- each run's summary report to `<run_dir>/<run_name>_summary.pdf` (if `plot_run_summaries = True`)

In [ ]:
lcurve_diagnostics = pyissm.inversion.sensitivity.compute_sensitivity_diagnostics(lcurve_manifest,
                                                                                  output_dir = sensitivity_dir + '/lcurve',
                                                                                  plot_run_summaries = True)
lcurve_diagnostics[['run_name', 'cf501', 'vel_rmse', 'cost_101', 'cost_103', 'cost_501', 'rmse_gradient_magnitude']]

### 3.4 Plot the L-curve

`pyissm.inversion.plot.plot_lcurve()` infers the regularisation term (`cost_501`), its coefficient (`cf501`) and the misfit terms (`cost_101` and `cost_103`) from the diagnostics table. The regularisation term is divided by its coefficient so that it measures only the roughness of the friction field. Each point is labelled with its regularisation coefficient.

In [ ]:
fig, ax = pyissm.inversion.plot.plot_lcurve(lcurve_diagnostics)
ax.set_title('L-curve: cost function 501')

Choose the coefficient closest to the corner of the curve. Below, we set it to `1e-19`, but **you should update this value based on your own L-curve**: the corner depends on the mesh, the observations and the misfit coefficients.

It is also useful to compare the inverted friction fields from either side of the corner. Below, we load the runs with the smallest, chosen and largest regularisation coefficients using `pyissm.inversion.sensitivity.load_parameter_sensitivity_run()`.

In [ ]:
# UPDATE based on your L-curve
alpha = 1e-19

# Compare inverted friction fields for the smallest, chosen and largest coefficients
rows = [lcurve_manifest.loc[lcurve_manifest['cf501'].idxmin()],
        lcurve_manifest.loc[np.isclose(lcurve_manifest['cf501'], alpha)].iloc[0],
        lcurve_manifest.loc[lcurve_manifest['cf501'].idxmax()]]

fig, axes = plt.subplots(1, 3, figsize = (15, 4.5), sharey = True, constrained_layout = True)
for i, (row, ax) in enumerate(zip(rows, axes)):
    mdi = pyissm.inversion.sensitivity.load_parameter_sensitivity_run(row)
    pyissm.plot.plot_model_field(mdi,
                                 mdi.results.StressbalanceSolution.FrictionCoefficient,
                                 ax = ax,
                                 show_cbar = True,
                                 vmin = 0,
                                 vmax = 200)
    ax.set_title(f"cf501 = {row['cf501']:.0e}")
for ax in axes[1:]:
    ax.set_ylabel('')

---
## 4. Balancing the misfit terms: sensitivity heatmaps

With the regularisation coefficient fixed, we now vary the two misfit coefficients together. The absolute misfit (101) is dominated by fast-flowing ice, while the log misfit (103) gives more weight to slow-flowing ice, so their relative weighting determines where the model fits the observations best.

### 4.1 Build and run a 2D parameter grid

We test three values for each misfit coefficient (9 runs). We use `initial_run_id = 101` so that these runs have different `run_id`s and `run_name`s from the L-curve runs, which keeps the two experiments distinct if you combine their manifests or diagnostics later.

In [ ]:
misfit_coefficients = {101: [10, 50, 200],
                       103: [0.1, 1, 10],
                       501: [alpha]}

misfit_grid = pyissm.inversion.sensitivity.build_parameter_grid(misfit_coefficients, initial_run_id = 101)
misfit_grid

In [ ]:
misfit_manifest = pyissm.inversion.sensitivity.parameter_sensitivity(md,
                                                                     misfit_grid,
                                                                     output_dir = sensitivity_dir + '/misfit',
                                                                     run = True,
                                                                     global_mask = global_mask,
                                                                     coeff_masks = coeff_masks)

misfit_diagnostics = pyissm.inversion.sensitivity.compute_sensitivity_diagnostics(misfit_manifest,
                                                                                  output_dir = sensitivity_dir + '/misfit',
                                                                                  plot_run_summaries = True)

<div class="alert alert-info">

**Reloading results later**

The manifest and diagnostics are saved as CSV files, so you can come back to a completed sensitivity study without re-running it:

```python
misfit_manifest = pd.read_csv(sensitivity_dir + '/misfit/manifest.csv')
misfit_diagnostics = pd.read_csv(sensitivity_dir + '/misfit/diagnostics.csv')
```

</div>

### 4.2 Plot sensitivity heatmaps

`pyissm.inversion.plot.plot_sensitivity_heatmap()` plots any diagnostic as a function of two coefficients. Below, we compare four diagnostics:

- `vel_rmse` — overall fit to the observations
- `ratio_101_103` — which misfit term dominates
- `positive_residual_fraction` — fraction of the domain where the model is too fast
- `rmse_gradient_magnitude` — roughness of the inverted friction field

In [ ]:
heatmap_values = ['vel_rmse', 'ratio_101_103', 'positive_residual_fraction', 'rmse_gradient_magnitude']

fig, axes = plt.subplots(2, 2, figsize = (12, 9), constrained_layout = True)
for value, ax in zip(heatmap_values, axes.flat):
    pyissm.inversion.plot.plot_sensitivity_heatmap(misfit_diagnostics,
                                                   x = 'cf101',
                                                   y = 'cf103',
                                                   value = value,
                                                   ax = ax)

---
## 5. Ranking runs and selecting a result

No single diagnostic identifies the "best" inversion, and the diagnostics have very different units and ranges. `pyissm.inversion.sensitivity.normalize_diagnostics()` rescales selected columns to the range [0, 1], where **1 is always best**. By default, lower values are treated as better; use `higher_is_better` to reverse this for individual columns.

Here, we score each run on its velocity RMSE and the smoothness of its friction field, plus how evenly balanced its positive and negative residuals are. We give each equal weight. This is only one possible scoring scheme: choose the diagnostics and weights that matter for your application.

In [ ]:
# How far the residuals are from an even split between too fast and too slow (0 = perfectly balanced)
misfit_diagnostics['residual_imbalance'] = np.abs(misfit_diagnostics['positive_residual_fraction'] -
                                                  misfit_diagnostics['negative_residual_fraction'])

score_columns = ['vel_rmse', 'rmse_gradient_magnitude', 'residual_imbalance']

ranked = pyissm.inversion.sensitivity.normalize_diagnostics(misfit_diagnostics, columns = score_columns)
ranked['score'] = ranked[[f'{col}_norm' for col in score_columns]].mean(axis = 1)
ranked = ranked.sort_values('score', ascending = False)

ranked[['run_name', 'cf101', 'cf103', 'score'] + score_columns]

In [ ]:
fig, ax = pyissm.inversion.plot.plot_sensitivity_heatmap(ranked, x = 'cf101', y = 'cf103', value = 'score')

Finally, we load the highest-scoring run and save it for use in subsequent simulations (for example, the Higher-Order extrusion in Section 5 of Example 2). The run's summary report is available at `<run_dir>/<run_name>_summary.pdf`.

In [ ]:
best = ranked.iloc[0]
print(f"Selected run: {best['run_name']} (cf101 = {best['cf101']:g}, cf103 = {best['cf103']:g}, cf501 = {best['cf501']:g})")
print(f"Summary report: {best['run_dir']}/{best['run_name']}_summary.pdf")

md_best = pyissm.inversion.sensitivity.load_parameter_sensitivity_run(best)
pyissm.model.io.save_model(md_best, tutorial_dir + '/ex5_PIG_control_drag_selected.nc')

In [ ]:
# Visualise the selected inversion
residuals = pyissm.tools.diagnostics.velocity_residuals(md_best)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize = (12, 5), sharey = True, constrained_layout = True)
pyissm.plot.plot_model_field(md_best, md_best.results.StressbalanceSolution.FrictionCoefficient, ax = ax1, show_cbar = True)
pyissm.plot.plot_model_field(md_best, residuals, ax = ax2, show_cbar = True, cmap = 'RdBu_r', vmin = -200, vmax = 200, ylabel = '')
ax1.set_title('Basal friction coefficient')
ax2.set_title('Velocity residual (Mod - Obs)')